In [45]:
import numpy as np
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix, classification_report, roc_curve
from sklearn.model_selection import GridSearchCV
import matplotlib.pyplot as plt
import seaborn as sns
import pickle

In [46]:

def load_data():
    train_latents=np.load('/kaggle/input/ct-seed-999-ct1/train_latents.npy')
    val_latents=np.load('/kaggle/input/ct-seed-999-ct1/val_latents.npy')
    test_latents=np.load('/kaggle/input/ct-seed-999-ct1/test_latents.npy')
    train_labels=np.load('/kaggle/input/ct-seed-999-ct1/train_labels.npy')
    val_labels=np.load('/kaggle/input/ct-seed-999-ct1/val_labels.npy')
    test_labels=np.load('/kaggle/input/ct-seed-999-ct1/test_labels.npy')
    return train_latents,val_latents,test_latents,train_labels,val_labels,test_labels

def evaluate_model(y_true,y_pred,y_pred_proba,model_name):
    acc=accuracy_score(y_true,y_pred)
    prec=precision_score(y_true,y_pred)
    rec=recall_score(y_true,y_pred)
    f1=f1_score(y_true,y_pred)
    auc=roc_auc_score(y_true,y_pred_proba)
    cm=confusion_matrix(y_true,y_pred)
    print(f"\n{model_name} Results:")
    print(f"Accuracy: {acc:.4f}")
    print(f"Precision: {prec:.4f}")
    print(f"Recall: {rec:.4f}")
    print(f"F1 Score: {f1:.4f}")
    print(f"AUC-ROC: {auc:.4f}")
    print(f"\nConfusion Matrix:")
    print(cm)
    print(f"\nClassification Report:")
    print(classification_report(y_true,y_pred,target_names=['Normal','COVID']))
    return {'accuracy':acc,'precision':prec,'recall':rec,'f1':f1,'auc':auc,'confusion_matrix':cm}

def plot_confusion_matrix(cm,model_name,save_path):
    plt.figure(figsize=(8,6))
    sns.heatmap(cm,annot=True,fmt='d',cmap='Blues',xticklabels=['Normal','COVID'],yticklabels=['Normal','COVID'])
    plt.title(f'{model_name} Confusion Matrix')
    plt.ylabel('True Label')
    plt.xlabel('Predicted Label')
    plt.tight_layout()
    plt.savefig(save_path,dpi=300,bbox_inches='tight')
    plt.close()

def plot_roc_curve(y_true,y_pred_proba,model_name,save_path):
    fpr,tpr,_=roc_curve(y_true,y_pred_proba)
    auc=roc_auc_score(y_true,y_pred_proba)
    plt.figure(figsize=(8,6))
    plt.plot(fpr,tpr,linewidth=2,label=f'{model_name} (AUC={auc:.4f})')
    plt.plot([0,1],[0,1],'k--',linewidth=1)
    plt.xlim([0.0,1.0])
    plt.ylim([0.0,1.05])
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.title(f'{model_name} ROC Curve')
    plt.legend(loc="lower right")
    plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(save_path,dpi=300,bbox_inches='tight')
    plt.close()

def plot_comparison(results,save_path):
    models=list(results.keys())
    metrics=['accuracy','precision','recall','f1','auc']
    fig,ax=plt.subplots(figsize=(12,6))
    x=np.arange(len(metrics))
    width=0.35
    for i,model in enumerate(models):
        values=[results[model][m] for m in metrics]
        ax.bar(x+i*width,values,width,label=model)
    ax.set_xlabel('Metrics')
    ax.set_ylabel('Score')
    ax.set_title('Model Comparison')
    ax.set_xticks(x+width/2)
    ax.set_xticklabels(metrics)
    ax.legend()
    ax.grid(axis='y',alpha=0.3)
    plt.tight_layout()
    plt.savefig(save_path,dpi=300,bbox_inches='tight')
    plt.close()

def train_rbf_svm(X_train,y_train,X_val,y_val):
    print("\nTraining RBF SVM...")
    scaler=StandardScaler()
    X_train_scaled=scaler.fit_transform(X_train)
    X_val_scaled=scaler.transform(X_val)
    svm=SVC(kernel='rbf',C=1.0,gamma='scale',probability=True,random_state=42)
    svm.fit(X_train_scaled,y_train)
    y_pred=svm.predict(X_val_scaled)
    y_pred_proba=svm.predict_proba(X_val_scaled)[:,1]
    return svm,scaler,y_pred,y_pred_proba

def train_logistic_regression(X_train,y_train,X_val,y_val):
    print("\nTraining Logistic Regression...")
    scaler=StandardScaler()
    X_train_scaled=scaler.fit_transform(X_train)
    X_val_scaled=scaler.transform(X_val)
    lr=LogisticRegression(max_iter=1000,random_state=42,solver='lbfgs')
    lr.fit(X_train_scaled,y_train)
    y_pred=lr.predict(X_val_scaled)
    y_pred_proba=lr.predict_proba(X_val_scaled)[:,1]
    return lr,scaler,y_pred,y_pred_proba

def train_linear_svm(X_train,y_train,X_val,y_val):
    print("\nTraining Linear SVM...")
    scaler=StandardScaler()
    X_train_scaled=scaler.fit_transform(X_train)
    X_val_scaled=scaler.transform(X_val)
    svm=SVC(kernel='linear',C=1.0,probability=True,random_state=42)
    svm.fit(X_train_scaled,y_train)
    y_pred=svm.predict(X_val_scaled)
    y_pred_proba=svm.predict_proba(X_val_scaled)[:,1]
    return svm,scaler,y_pred,y_pred_proba

In [47]:
train_latents , val_latents, test_latents , train_labels , val_labels , test_labels = load_data()
all_latents = np.vstack([train_latents, val_latents, test_latents])
print(f"Shape: {all_latents.shape}")
print(f"\nGlobal Statistics:")
print(f"  Min value:  {all_latents.min():.6f}")
print(f"  Max value:  {all_latents.max():.6f}")
print(f"  Mean:       {all_latents.mean():.6f}")
print(f"  Std:        {all_latents.std():.6f}")
print(f"  Median:     {np.median(all_latents):.6f}")

Shape: (5500, 85)

Global Statistics:
  Min value:  -61.886192
  Max value:  70.476814
  Mean:       0.102131
  Std:        8.662813
  Median:     0.081030


In [48]:
def main():
    X_train,X_val,X_test,y_train,y_val,y_test=load_data()
    print(f"Train: {X_train.shape}, Val: {X_val.shape}, Test: {X_test.shape}")
    results_val={}
    results_test={}
    models={}
    scalers={}
    svm_rbf,scaler_rbf,y_pred_rbf,y_pred_proba_rbf=train_rbf_svm(X_train,y_train,X_val,y_val)
    results_val['RBF_SVM']=evaluate_model(y_val,y_pred_rbf,y_pred_proba_rbf,'RBF SVM (Validation)')
    plot_confusion_matrix(results_val['RBF_SVM']['confusion_matrix'],'RBF SVM','rbf_svm_cm_val.png')
    plot_roc_curve(y_val,y_pred_proba_rbf,'RBF SVM','rbf_svm_roc_val.png')
    models['rbf_svm']=svm_rbf
    scalers['rbf_svm']=scaler_rbf
    lr,scaler_lr,y_pred_lr,y_pred_proba_lr=train_logistic_regression(X_train,y_train,X_val,y_val)
    results_val['LogReg']=evaluate_model(y_val,y_pred_lr,y_pred_proba_lr,'Logistic Regression (Validation)')
    plot_confusion_matrix(results_val['LogReg']['confusion_matrix'],'Logistic Regression','logreg_cm_val.png')
    plot_roc_curve(y_val,y_pred_proba_lr,'Logistic Regression','logreg_roc_val.png')
    models['logreg']=lr
    scalers['logreg']=scaler_lr
    svm_linear,scaler_linear,y_pred_linear,y_pred_proba_linear=train_linear_svm(X_train,y_train,X_val,y_val)
    results_val['Linear_SVM']=evaluate_model(y_val,y_pred_linear,y_pred_proba_linear,'Linear SVM (Validation)')
    plot_confusion_matrix(results_val['Linear_SVM']['confusion_matrix'],'Linear SVM','linear_svm_cm_val.png')
    plot_roc_curve(y_val,y_pred_proba_linear,'Linear SVM','linear_svm_roc_val.png')
    models['linear_svm']=svm_linear
    scalers['linear_svm']=scaler_linear
    plot_comparison(results_val,'model_comparison_val.png')
    X_test_rbf=scalers['rbf_svm'].transform(X_test)
    y_pred_test_rbf=models['rbf_svm'].predict(X_test_rbf)
    y_pred_proba_test_rbf=models['rbf_svm'].predict_proba(X_test_rbf)[:,1]
    results_test['RBF_SVM']=evaluate_model(y_test,y_pred_test_rbf,y_pred_proba_test_rbf,'RBF SVM (Test)')
    plot_confusion_matrix(results_test['RBF_SVM']['confusion_matrix'],'RBF SVM','rbf_svm_cm_test.png')
    plot_roc_curve(y_test,y_pred_proba_test_rbf,'RBF SVM','rbf_svm_roc_test.png')
    X_test_lr=scalers['logreg'].transform(X_test)
    y_pred_test_lr=models['logreg'].predict(X_test_lr)
    y_pred_proba_test_lr=models['logreg'].predict_proba(X_test_lr)[:,1]
    results_test['LogReg']=evaluate_model(y_test,y_pred_test_lr,y_pred_proba_test_lr,'Logistic Regression (Test)')
    plot_confusion_matrix(results_test['LogReg']['confusion_matrix'],'Logistic Regression','logreg_cm_test.png')
    plot_roc_curve(y_test,y_pred_proba_test_lr,'Logistic Regression','logreg_roc_test.png')
    X_test_linear=scalers['linear_svm'].transform(X_test)
    y_pred_test_linear=models['linear_svm'].predict(X_test_linear)
    y_pred_proba_test_linear=models['linear_svm'].predict_proba(X_test_linear)[:,1]
    results_test['Linear_SVM']=evaluate_model(y_test,y_pred_test_linear,y_pred_proba_test_linear,'Linear SVM (Test)')
    plot_confusion_matrix(results_test['Linear_SVM']['confusion_matrix'],'Linear SVM','linear_svm_cm_test.png')
    plot_roc_curve(y_test,y_pred_proba_test_linear,'Linear SVM','linear_svm_roc_test.png')
    plot_comparison(results_test,'model_comparison_test.png')
    with open('rbf_svm_model.pkl','wb') as f:
        pickle.dump({'model':models['rbf_svm'],'scaler':scalers['rbf_svm']},f)
    with open('logreg_model.pkl','wb') as f:
        pickle.dump({'model':models['logreg'],'scaler':scalers['logreg']},f)
    with open('linear_svm_model.pkl','wb') as f:
        pickle.dump({'model':models['linear_svm'],'scaler':scalers['linear_svm']},f)
    for model_name,metrics in results_val.items():
        print(f"{model_name}: Acc={metrics['accuracy']:.4f} F1={metrics['f1']:.4f} AUC={metrics['auc']:.4f}")
    print("\nTest Set:")
    for model_name,metrics in results_test.items():
        print(f"{model_name}: Acc={metrics['accuracy']:.4f} F1={metrics['f1']:.4f} AUC={metrics['auc']:.4f}")
if __name__=='__main__':
    main()

Train: (3878, 85), Val: (804, 85), Test: (818, 85)

Training RBF SVM...

RBF SVM (Validation) Results:
Accuracy: 0.6493
Precision: 0.6229
Recall: 0.7387
F1 Score: 0.6759
AUC-ROC: 0.6803

Confusion Matrix:
[[228 178]
 [104 294]]

Classification Report:
              precision    recall  f1-score   support

      Normal       0.69      0.56      0.62       406
       COVID       0.62      0.74      0.68       398

    accuracy                           0.65       804
   macro avg       0.65      0.65      0.65       804
weighted avg       0.66      0.65      0.65       804


Training Logistic Regression...

Logistic Regression (Validation) Results:
Accuracy: 0.5958
Precision: 0.5680
Recall: 0.7663
F1 Score: 0.6524
AUC-ROC: 0.6408

Confusion Matrix:
[[174 232]
 [ 93 305]]

Classification Report:
              precision    recall  f1-score   support

      Normal       0.65      0.43      0.52       406
       COVID       0.57      0.77      0.65       398

    accuracy                    

In [49]:
import pandas as pd 

In [50]:
def get_feature_importance_subset(model, scaler, feature_names, feature_indices=None):

    # Get all coefficients from the model
    all_coefficients = model.coef_[0]
    
    if feature_indices is None:
        # If no indices provided, assume the features are the first N features
        feature_indices = list(range(len(feature_names)))
    
    # Extract coefficients for the specified features
    coefficients = all_coefficients[feature_indices]
    
    # Create DataFrame with feature names and coefficients
    feature_importance = pd.DataFrame({
        'Feature': feature_names,
        'Coefficient': coefficients,
        'Abs_Coefficient': np.abs(coefficients)
    })
    
    # Sort by absolute coefficient value
    feature_importance = feature_importance.sort_values('Abs_Coefficient', ascending=False)
    
    return feature_importance

def plot_feature_importance_physics(feature_importance, filename='physics_feature_importance.png'):
    """
    Plot feature importance for physics features.
    """
    plt.figure(figsize=(10, 6))
    colors = ['red' if x < 0 else 'blue' for x in feature_importance['Coefficient']]
    
    plt.barh(feature_importance['Feature'], feature_importance['Coefficient'], color=colors)
    plt.xlabel('Coefficient Value', fontsize=12)
    plt.ylabel('Physics Features', fontsize=12)
    plt.title('Logistic Regression - Physics Feature Importance (14 Features)', 
              fontsize=14, fontweight='bold')
    plt.axvline(x=0, color='black', linestyle='--', linewidth=0.8)
    plt.grid(axis='x', alpha=0.3)
    
    # Add legend
    from matplotlib.patches import Patch
    legend_elements = [Patch(facecolor='blue', label='Positive Impact'),
                      Patch(facecolor='red', label='Negative Impact')]
    plt.legend(handles=legend_elements, loc='lower right')
    
    plt.tight_layout()
    plt.savefig(filename, dpi=300, bbox_inches='tight')
    plt.close()
    print(f"Physics feature importance plot saved to {filename}")

def main():
    # Define the 14 physics feature names
    physics_feature_names = ['mean_HU', 'HU_std', 'HU_p10', 'HU_p25', 'HU_p50', 
                             'HU_p75', 'HU_p90', 'mask_area', 'mask_frac', 'grad_mean', 
                             'grad_std', 'contrast', 'homog', 'entropy']
    
    # Indices for the 14 physics features (assuming they are the first 14 columns)
    physics_feature_indices = list(range(14))
    
    # Load data
    X_train, X_val, X_test, y_train, y_val, y_test = load_data()
    print(f"Train: {X_train.shape}, Val: {X_val.shape}, Test: {X_test.shape}")
    print(f"Total features in dataset: {X_train.shape[1]}")
    print(f"Physics features to analyze: {len(physics_feature_names)}\n")
    
    results_val = {}
    results_test = {}
    models = {}
    scalers = {}
    
    # Train RBF SVM
    svm_rbf, scaler_rbf, y_pred_rbf, y_pred_proba_rbf = train_rbf_svm(X_train, y_train, X_val, y_val)
    results_val['RBF_SVM'] = evaluate_model(y_val, y_pred_rbf, y_pred_proba_rbf, 'RBF SVM (Validation)')
    plot_confusion_matrix(results_val['RBF_SVM']['confusion_matrix'], 'RBF SVM', 'rbf_svm_cm_val.png')
    plot_roc_curve(y_val, y_pred_proba_rbf, 'RBF SVM', 'rbf_svm_roc_val.png')
    models['rbf_svm'] = svm_rbf
    scalers['rbf_svm'] = scaler_rbf
    
    # Train Logistic Regression
    lr, scaler_lr, y_pred_lr, y_pred_proba_lr = train_logistic_regression(X_train, y_train, X_val, y_val)
    results_val['LogReg'] = evaluate_model(y_val, y_pred_lr, y_pred_proba_lr, 'Logistic Regression (Validation)')
    plot_confusion_matrix(results_val['LogReg']['confusion_matrix'], 'Logistic Regression', 'logreg_cm_val.png')
    plot_roc_curve(y_val, y_pred_proba_lr, 'Logistic Regression', 'logreg_roc_val.png')
    models['logreg'] = lr
    scalers['logreg'] = scaler_lr
    
    print("\n" + "="*70)
    print("PHYSICS FEATURES IMPORTANCE (14 Features)")
    print("="*70)
    
    physics_importance = get_feature_importance_subset(
        lr, scaler_lr, physics_feature_names, physics_feature_indices
    )
    
    print("\nAll 14 Physics Features Ranked by Importance:")
    print(physics_importance.to_string(index=False))
    
    print("\n" + "-"*70)
    print("Top 5 Most Important Physics Features:")
    print("-"*70)
    for idx, row in physics_importance.head(5).iterrows():
        impact = "increases" if row['Coefficient'] > 0 else "decreases"
        print(f"{row['Feature']:15s}: {row['Coefficient']:8.4f} ({impact} positive class probability)")
    
    print("\n" + "-"*70)
    print("Bottom 5 Least Important Physics Features:")
    print("-"*70)
    for idx, row in physics_importance.tail(5).iterrows():
        impact = "increases" if row['Coefficient'] > 0 else "decreases"
        print(f"{row['Feature']:15s}: {row['Coefficient']:8.4f} ({impact} positive class probability)")
    
    plot_feature_importance_physics(physics_importance, 'physics_feature_importance.png')
    physics_importance.to_csv('physics_feature_importance.csv', index=False)
    print("\nPhysics feature importance saved to 'physics_feature_importance.csv'")
    print("="*70 + "\n")
    
    # ========== ALL FEATURES IMPORTANCE (Optional) ==========
    print("\n" + "="*70)
    print("ALL FEATURES IMPORTANCE (85 Features)")
    print("="*70)
    
    # Generate names for all 85 features
    all_feature_names = physics_feature_names + [f'Feature_{i}' for i in range(14, 85)]
    all_importance = get_feature_importance_subset(lr, scaler_lr, all_feature_names, list(range(85)))
    
    print("\nTop 10 Most Important Features (All 85):")
    print(all_importance.head(10).to_string(index=False))
    
    print("\n" + "-"*70)
    print("Physics Features in Top 20 Overall:")
    print("-"*70)
    top_20 = all_importance.head(20)
    physics_in_top_20 = top_20[top_20['Feature'].isin(physics_feature_names)]
    if len(physics_in_top_20) > 0:
        print(physics_in_top_20.to_string(index=False))
    else:
        print("No physics features in top 20")
    
    all_importance.to_csv('all_features_importance.csv', index=False)
    print("\nAll features importance saved to 'all_features_importance.csv'")
    print("="*70 + "\n")
    
    # Train Linear SVM
    svm_linear, scaler_linear, y_pred_linear, y_pred_proba_linear = train_linear_svm(X_train, y_train, X_val, y_val)
    results_val['Linear_SVM'] = evaluate_model(y_val, y_pred_linear, y_pred_proba_linear, 'Linear SVM (Validation)')
    plot_confusion_matrix(results_val['Linear_SVM']['confusion_matrix'], 'Linear SVM', 'linear_svm_cm_val.png')
    plot_roc_curve(y_val, y_pred_proba_linear, 'Linear SVM', 'linear_svm_roc_val.png')
    models['linear_svm'] = svm_linear
    scalers['linear_svm'] = scaler_linear
    
    # Plot validation comparison
    plot_comparison(results_val, 'model_comparison_val.png')
    
    # Test RBF SVM
    X_test_rbf = scalers['rbf_svm'].transform(X_test)
    y_pred_test_rbf = models['rbf_svm'].predict(X_test_rbf)
    y_pred_proba_test_rbf = models['rbf_svm'].predict_proba(X_test_rbf)[:, 1]
    results_test['RBF_SVM'] = evaluate_model(y_test, y_pred_test_rbf, y_pred_proba_test_rbf, 'RBF SVM (Test)')
    plot_confusion_matrix(results_test['RBF_SVM']['confusion_matrix'], 'RBF SVM', 'rbf_svm_cm_test.png')
    plot_roc_curve(y_test, y_pred_proba_test_rbf, 'RBF SVM', 'rbf_svm_roc_test.png')
    
    # Test Logistic Regression
    X_test_lr = scalers['logreg'].transform(X_test)
    y_pred_test_lr = models['logreg'].predict(X_test_lr)
    y_pred_proba_test_lr = models['logreg'].predict_proba(X_test_lr)[:, 1]
    results_test['LogReg'] = evaluate_model(y_test, y_pred_test_lr, y_pred_proba_test_lr, 'Logistic Regression (Test)')
    plot_confusion_matrix(results_test['LogReg']['confusion_matrix'], 'Logistic Regression', 'logreg_cm_test.png')
    plot_roc_curve(y_test, y_pred_proba_test_lr, 'Logistic Regression', 'logreg_roc_test.png')
    
    # Test Linear SVM
    X_test_linear = scalers['linear_svm'].transform(X_test)
    y_pred_test_linear = models['linear_svm'].predict(X_test_linear)
    y_pred_proba_test_linear = models['linear_svm'].predict_proba(X_test_linear)[:, 1]
    results_test['Linear_SVM'] = evaluate_model(y_test, y_pred_test_linear, y_pred_proba_test_linear, 'Linear SVM (Test)')
    plot_confusion_matrix(results_test['Linear_SVM']['confusion_matrix'], 'Linear SVM', 'linear_svm_cm_test.png')
    plot_roc_curve(y_test, y_pred_proba_test_linear, 'Linear SVM', 'linear_svm_roc_test.png')
    
    # Plot test comparison
    plot_comparison(results_test, 'model_comparison_test.png')
    
    # Save models
    with open('rbf_svm_model.pkl', 'wb') as f:
        pickle.dump({'model': models['rbf_svm'], 'scaler': scalers['rbf_svm']}, f)
    
    with open('logreg_model.pkl', 'wb') as f:
        pickle.dump({'model': models['logreg'], 'scaler': scalers['logreg']}, f)
    
    with open('linear_svm_model.pkl', 'wb') as f:
        pickle.dump({'model': models['linear_svm'], 'scaler': scalers['linear_svm']}, f)
    
    # Print summary
    print("\n" + "="*60)
    print("VALIDATION SET RESULTS:")
    print("="*60)
    for model_name, metrics in results_val.items():
        print(f"{model_name:15s}: Acc={metrics['accuracy']:.4f} F1={metrics['f1']:.4f} AUC={metrics['auc']:.4f}")
    
    print("\n" + "="*60)
    print("TEST SET RESULTS:")
    print("="*60)
    for model_name, metrics in results_test.items():
        print(f"{model_name:15s}: Acc={metrics['accuracy']:.4f} F1={metrics['f1']:.4f} AUC={metrics['auc']:.4f}")
    print("="*60)

if __name__ == '__main__':
    main()

Train: (3878, 85), Val: (804, 85), Test: (818, 85)
Total features in dataset: 85
Physics features to analyze: 14


Training RBF SVM...

RBF SVM (Validation) Results:
Accuracy: 0.6493
Precision: 0.6229
Recall: 0.7387
F1 Score: 0.6759
AUC-ROC: 0.6803

Confusion Matrix:
[[228 178]
 [104 294]]

Classification Report:
              precision    recall  f1-score   support

      Normal       0.69      0.56      0.62       406
       COVID       0.62      0.74      0.68       398

    accuracy                           0.65       804
   macro avg       0.65      0.65      0.65       804
weighted avg       0.66      0.65      0.65       804


Training Logistic Regression...

Logistic Regression (Validation) Results:
Accuracy: 0.5958
Precision: 0.5680
Recall: 0.7663
F1 Score: 0.6524
AUC-ROC: 0.6408

Confusion Matrix:
[[174 232]
 [ 93 305]]

Classification Report:
              precision    recall  f1-score   support

      Normal       0.65      0.43      0.52       406
       COVID       0.57 

In [51]:
def load_data_14dim_only():
    
    # Extract ONLY first 14 columns (predicted attributes)
    train_latents_14 = train_latents[:, :14]
    val_latents_14 = val_latents[:, :14]
    test_latents_14 = test_latents[:, :14]
    
    return train_latents_14, val_latents_14, test_latents_14, train_labels, val_labels, test_labels


def main_14dim_comparison():
    X_train_85, X_val_85, X_test_85, y_train, y_val, y_test = load_data()
    
    X_train_14, X_val_14, X_test_14, _, _, _ = load_data_14dim_only()
    
    print(f"85-dim Latent Space: {X_train_85.shape}")
    print(f"14-dim Attributes:   {X_train_14.shape}\n")
    results_85 = {}
    
    # Linear SVM on 85-dim
    print(" Linear SVM on 85-dim...")
    svm_85, scaler_85, pred_85, proba_85 = train_linear_svm(
        X_train_85, y_train, X_val_85, y_val
    )
    results_85['Linear_SVM'] = evaluate_model(y_val, pred_85, proba_85, 'Linear SVM (85-dim)')
    
    # Logistic Regression on 85-dim
    print(" Logistic Regression on 85-dim...")
    lr_85, scaler_lr_85, pred_lr_85, proba_lr_85 = train_logistic_regression(
        X_train_85, y_train, X_val_85, y_val
    )
    results_85['LogReg'] = evaluate_model(y_val, pred_lr_85, proba_lr_85, 'Logistic Regression (85-dim)')
    
    # RBF SVM on 85-dim
    print(" RBF SVM on 85-dim...")
    rbf_85, scaler_rbf_85, pred_rbf_85, proba_rbf_85 = train_rbf_svm(
        X_train_85, y_train, X_val_85, y_val
    )
    results_85['RBF_SVM'] = evaluate_model(y_val, pred_rbf_85, proba_rbf_85, 'RBF SVM (85-dim)')
    
    results_14 = {}
    
    # Linear SVM on 14-dim
    print("Linear SVM on 14-dim...")
    svm_14, scaler_14, pred_14, proba_14 = train_linear_svm(
        X_train_14, y_train, X_val_14, y_val
    )
    results_14['Linear_SVM'] = evaluate_model(y_val, pred_14, proba_14, 'Linear SVM (14-dim)')
    
    # Logistic Regression on 14-dim
    print(" Logistic Regression on 14-dim...")
    lr_14, scaler_lr_14, pred_lr_14, proba_lr_14 = train_logistic_regression(
        X_train_14, y_train, X_val_14, y_val
    )
    results_14['LogReg'] = evaluate_model(y_val, pred_lr_14, proba_lr_14, 'Logistic Regression (14-dim)')
    
    # RBF SVM on 14-dim
    print(" RBF SVM on 14-dim...")
    rbf_14, scaler_rbf_14, pred_rbf_14, proba_rbf_14 = train_rbf_svm(
        X_train_14, y_train, X_val_14, y_val
    )
    results_14['RBF_SVM'] = evaluate_model(y_val, pred_rbf_14, proba_rbf_14, 'RBF SVM (14-dim)')
    
    X_test_85_scaled = scaler_lr_85.transform(X_test_85)
    pred_test_85 = lr_85.predict(X_test_85_scaled)
    proba_test_85 = lr_85.predict_proba(X_test_85_scaled)[:, 1]
    test_85 = evaluate_model(y_test, pred_test_85, proba_test_85, 'LogReg 85-dim (Test)')
    
    # Test 14-dim
    X_test_14_scaled = scaler_lr_14.transform(X_test_14)
    pred_test_14 = lr_14.predict(X_test_14_scaled)
    proba_test_14 = lr_14.predict_proba(X_test_14_scaled)[:, 1]
    test_14 = evaluate_model(y_test, pred_test_14, proba_test_14, 'LogReg 14-dim (Test)')
    

    
    comparison_df = pd.DataFrame({
        'Classifier': ['Linear SVM', 'Logistic Regression', 'RBF SVM'],
        '85-dim Acc': [results_85['Linear_SVM']['accuracy'], 
                       results_85['LogReg']['accuracy'], 
                       results_85['RBF_SVM']['accuracy']],
        '14-dim Acc': [results_14['Linear_SVM']['accuracy'], 
                       results_14['LogReg']['accuracy'], 
                       results_14['RBF_SVM']['accuracy']],
        '85-dim AUC': [results_85['Linear_SVM']['auc'], 
                       results_85['LogReg']['auc'], 
                       results_85['RBF_SVM']['auc']],
        '14-dim AUC': [results_14['Linear_SVM']['auc'], 
                       results_14['LogReg']['auc'], 
                       results_14['RBF_SVM']['auc']],
    })
    
    # Calculate performance gap
    comparison_df['Acc Gap (%)'] = (comparison_df['85-dim Acc'] - comparison_df['14-dim Acc']) * 100
    comparison_df['AUC Gap'] = comparison_df['85-dim AUC'] - comparison_df['14-dim AUC']
    
    print("\nValidation Set Performance:")
    print(comparison_df.to_string(index=False))
    
    print(f"\nTest Set Performance (Logistic Regression):")
    print(f"  85-dim: Acc={test_85['accuracy']:.4f}, AUC={test_85['auc']:.4f}")
    print(f"  14-dim: Acc={test_14['accuracy']:.4f}, AUC={test_14['auc']:.4f}")
    print(f"  Gap:    {(test_85['accuracy'] - test_14['accuracy'])*100:.2f}% accuracy, {test_85['auc'] - test_14['auc']:.4f} AUC")
    
    # Save comparison
    comparison_df.to_csv('85dim_vs_14dim_comparison.csv', index=False)
    print(f"\n✓ Comparison saved to '85dim_vs_14dim_comparison.csv'")
    
    # ========== VISUALIZATION ==========
    plot_comparison_side_by_side(results_85, results_14, 'comparison_85_vs_14.png')
    
    avg_gap = comparison_df['Acc Gap (%)'].mean()
    print(f"Average accuracy gap: {avg_gap:.2f}%")
    print(f"Dimensionality reduction: {(1 - 14/85)*100:.1f}% fewer features")
    print(f"Interpretability gain: 14 physics features vs 85 black-box dimensions")
    print("="*70 + "\n")
    
    return comparison_df, results_85, results_14




In [52]:
def plot_comparison_side_by_side(results_85, results_14, save_path):
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    
    models = ['Linear_SVM', 'LogReg', 'RBF_SVM']
    metrics = ['accuracy', 'precision', 'recall', 'f1', 'auc']
    
    # 85-dim results
    ax = axes[0]
    x = np.arange(len(metrics))
    width = 0.25
    for i, model in enumerate(models):
        values = [results_85[model][m] for m in metrics]
        ax.bar(x + i*width, values, width, label=model.replace('_', ' '))
    
    ax.set_xlabel('Metrics', fontsize=12)
    ax.set_ylabel('Score', fontsize=12)
    ax.set_title('85-Dimensional Full Latent Space', fontsize=14, fontweight='bold')
    ax.set_xticks(x + width)
    ax.set_xticklabels(metrics)
    ax.legend()
    ax.grid(axis='y', alpha=0.3)
    ax.set_ylim([0, 1])
    
    # 14-dim results
    ax = axes[1]
    for i, model in enumerate(models):
        values = [results_14[model][m] for m in metrics]
        ax.bar(x + i*width, values, width, label=model.replace('_', ' '))
    
    ax.set_xlabel('Metrics', fontsize=12)
    ax.set_ylabel('Score', fontsize=12)
    ax.set_title('14-Dimensional Predicted Attributes', fontsize=14, fontweight='bold')
    ax.set_xticks(x + width)
    ax.set_xticklabels(metrics)
    ax.legend()
    ax.grid(axis='y', alpha=0.3)
    ax.set_ylim([0, 1])
    
    plt.suptitle('Classification Performance: 85-dim vs 14-dim', 
                 fontsize=16, fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.close()
    print(f"Comparison plot saved to {save_path}")


if __name__ == '__main__':
    # Run the comparison
    comparison_df, results_85, results_14 = main_14dim_comparison()

85-dim Latent Space: (3878, 85)
14-dim Attributes:   (3878, 14)

 Linear SVM on 85-dim...

Training Linear SVM...

Linear SVM (85-dim) Results:
Accuracy: 0.5958
Precision: 0.5701
Recall: 0.7462
F1 Score: 0.6464
AUC-ROC: 0.6439

Confusion Matrix:
[[182 224]
 [101 297]]

Classification Report:
              precision    recall  f1-score   support

      Normal       0.64      0.45      0.53       406
       COVID       0.57      0.75      0.65       398

    accuracy                           0.60       804
   macro avg       0.61      0.60      0.59       804
weighted avg       0.61      0.60      0.59       804

 Logistic Regression on 85-dim...

Training Logistic Regression...

Logistic Regression (85-dim) Results:
Accuracy: 0.5958
Precision: 0.5680
Recall: 0.7663
F1 Score: 0.6524
AUC-ROC: 0.6408

Confusion Matrix:
[[174 232]
 [ 93 305]]

Classification Report:
              precision    recall  f1-score   support

      Normal       0.65      0.43      0.52       406
       COVID    

In [53]:
## import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score
from scipy.stats import pearsonr


def prepare_feature_dataframe(latents, labels, feature_names=None):
    """
    Convert latent representations to DataFrame with proper feature names
    """
    if feature_names is None:
        # Create feature names: 14 physics + 71 learned
        physics_features = [
            'HU_p50', 'entropy', 'HU_p10', 'HU_p90', 'HU_p25', 
            'HU_p75', 'HU_std', 'grad_std', 'mask_frac', 'mask_area',
            'homog', 'mean_HU', 'grad_mean', 'contrast'
        ]
        learned_features = [f'Feature_{i}' for i in range(14, 85)]
        feature_names = physics_features + learned_features
    
    df = pd.DataFrame(latents, columns=feature_names)
    df['label'] = labels
    df['class'] = df['label'].map({0: 'Normal', 1: 'COVID'})
    
    return df, feature_names[:14], feature_names[14:]



def analyze_feature_correlations(df, physics_features, learned_features, 
                                 top_n=10, save_path='correlation_analysis.png'):
    """
    Compute and visualize correlations between physics and learned features
    """
    print("\n" + "="*70)
    print("CORRELATION ANALYSIS: Physics vs Learned Features")
    print("="*70)
    

    top_learned = learned_features[:top_n]  
    
    # Compute correlation matrix
    all_features = physics_features + top_learned
    corr_matrix = df[all_features].corr()
    
    # Extract physics vs learned correlations
    corr_subset = corr_matrix.loc[physics_features, top_learned]
    
    # Plot heatmap
    fig, ax = plt.subplots(figsize=(12, 8))
    sns.heatmap(corr_subset, annot=True, cmap='coolwarm', center=0,
                vmin=-1, vmax=1, fmt='.2f', cbar_kws={'label': 'Correlation'})
    plt.title('Correlation: Physics Features vs Top 10 Learned Features', 
              fontsize=14, pad=20)
    plt.xlabel('Learned Features', fontsize=12)
    plt.ylabel('Physics Features', fontsize=12)
    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.close()
    
    # Print strongest correlations
    print("\nStrongest Correlations (|r| > 0.5):")
    for learned_feat in top_learned:
        strong_corrs = corr_subset[learned_feat][abs(corr_subset[learned_feat]) > 0.5]
        if len(strong_corrs) > 0:
            print(f"\n{learned_feat}:")
            for phys_feat, corr_val in strong_corrs.items():
                print(f"  {phys_feat}: r = {corr_val:.3f}")
    
    return corr_subset




def explain_learned_feature(df, target_feature, physics_features, max_degree=2):
    X = df[physics_features].values
    y = df[target_feature].values
    
    results = {}
    
    # Linear model
    lr_linear = LinearRegression()
    lr_linear.fit(X, y)
    y_pred_linear = lr_linear.predict(X)
    r2_linear = r2_score(y, y_pred_linear)
    results['r2_linear'] = r2_linear
    
    # Polynomial model (interactions)
    poly = PolynomialFeatures(degree=max_degree, include_bias=False)
    X_poly = poly.fit_transform(X)
    lr_poly = LinearRegression()
    lr_poly.fit(X_poly, y)
    y_pred_poly = lr_poly.predict(X_poly)
    r2_poly = r2_score(y, y_pred_poly)
    results['r2_poly'] = r2_poly
    
    # Get top contributing terms
    feature_names = poly.get_feature_names_out(physics_features)
    coef_df = pd.DataFrame({
        'term': feature_names,
        'coefficient': lr_poly.coef_,
        'abs_coefficient': np.abs(lr_poly.coef_)
    }).sort_values('abs_coefficient', ascending=False)
    
    results['top_terms'] = coef_df.head(5)
    results['delta_r2'] = r2_poly - r2_linear
    
    return results


def analyze_top_learned_features(df, physics_features, top_learned_features, 
                                 save_path='feature_explanation.txt'):
    results = {}
    with open(save_path, 'w') as f:
        for learned_feat in top_learned_features:
            print(f"\n{learned_feat}:")
            f.write(f"\n{'='*60}\n{learned_feat}\n{'='*60}\n")
            
            result = explain_learned_feature(df, learned_feat, physics_features)
            results[learned_feat] = result
            
            # Print and save results
            output = f"""
R² (linear combination):     {result['r2_linear']:.3f}
R² (with interactions):      {result['r2_poly']:.3f}
Improvement from interactions: {result['delta_r2']:.3f}

Top 5 Contributing Terms:
{result['top_terms'].to_string(index=False)}
"""
            print(output)
            f.write(output + "\n")
            
            # Interpretation
            if result['r2_poly'] < 0.3:
                interp = "→ NOVEL: Cannot be explained by physics features"
            elif result['r2_poly'] > 0.7:
                interp = "→ REDUNDANT: Mostly explained by physics features"
            else:
                interp = "→ INTERACTION: Captures combinations of physics features"
            
            print(interp)
            f.write(interp + "\n")
    
    return results

def compute_overlap_coefficient(dist1, dist2, bins=50):
    """
    Compute overlap coefficient between two distributions
    """
    hist1, bin_edges = np.histogram(dist1, bins=bins, density=True)
    hist2, _ = np.histogram(dist2, bins=bin_edges, density=True)
    
    # Normalize to probability distributions
    hist1 = hist1 / hist1.sum()
    hist2 = hist2 / hist2.sum()
    
    # Overlap is the minimum at each bin
    overlap = np.minimum(hist1, hist2).sum()
    
    return overlap


def compute_cohens_d(group1, group2):
    """
    Compute Cohen's d effect size
    """
    n1, n2 = len(group1), len(group2)
    var1, var2 = np.var(group1, ddof=1), np.var(group2, ddof=1)
    pooled_std = np.sqrt(((n1-1)*var1 + (n2-1)*var2) / (n1+n2-2))
    return (np.mean(group1) - np.mean(group2)) / pooled_std


def analyze_feature_overlap(df, features_to_analyze, save_path='overlap_analysis.png'): 
    normal_data = df[df['label'] == 0]
    covid_data = df[df['label'] == 1]
    
    results = []
    
    for feature in features_to_analyze:
        overlap = compute_overlap_coefficient(
            normal_data[feature].values,
            covid_data[feature].values
        )
        cohens_d = compute_cohens_d(
            normal_data[feature].values,
            covid_data[feature].values
        )
        
        results.append({
            'feature': feature,
            'overlap_coef': overlap,
            'cohens_d': abs(cohens_d)
        })
    
    results_df = pd.DataFrame(results).sort_values('overlap_coef', ascending=False)
    
    # Print results
    print("\nFeature Overlap Analysis:")
    print(results_df.to_string(index=False))
    
    # Save to CSV
    results_df.to_csv('overlap_coefficients.csv', index=False)
    
    # Visualize top features with highest overlap
    fig, axes = plt.subplots(3, 3, figsize=(15, 12))
    axes = axes.flatten()
    
    for i, feature in enumerate(results_df['feature'].head(9)):
        ax = axes[i]
        
        # Plot distributions
        ax.hist(normal_data[feature], bins=30, alpha=0.5, 
                label='Normal', density=True, color='blue')
        ax.hist(covid_data[feature], bins=30, alpha=0.5, 
                label='COVID', density=True, color='red')
        
        # Add statistics
        overlap = results_df[results_df['feature']==feature]['overlap_coef'].values[0]
        cohens_d = results_df[results_df['feature']==feature]['cohens_d'].values[0]
        
        ax.set_title(f'{feature}\nOverlap={overlap:.3f}, Cohen\'s d={cohens_d:.2f}', 
                    fontsize=10)
        ax.legend()
        ax.set_xlabel('Feature Value')
        ax.set_ylabel('Density')
    
    plt.suptitle('Feature Distributions: Normal vs COVID\n' + 
                 'High Overlap = Data Ceiling', fontsize=14, y=0.995)
    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.close()
    
    # KEY FINDING for paper
    mean_overlap = results_df['overlap_coef'].mean()
    print(f"\n{'='*70}")
    print(f"KEY FINDING: Mean overlap coefficient = {mean_overlap:.3f}")
    
    return results_df

def run_complete_analysis(train_latents, val_latents, train_labels, val_labels,
                         lr_model, scaler):
    # Combine train + val for analysis
    all_latents = np.vstack([train_latents, val_latents])
    all_labels = np.concatenate([train_labels, val_labels])
    
    # Prepare DataFrame
    df, physics_features, learned_features = prepare_feature_dataframe(
        all_latents, all_labels
    )
    
    print("\n" + "="*70)
    print("DATA-FM ANALYSIS PIPELINE")
    print("="*70)
    print(f"Total samples: {len(df)}")
    print(f"Physics features: {len(physics_features)}")
    print(f"Learned features: {len(learned_features)}")
    
    # Get feature importance from logistic regression
    feature_importance = pd.DataFrame({
        'feature': physics_features + learned_features,
        'coefficient': lr_model.coef_[0],
        'abs_coefficient': np.abs(lr_model.coef_[0])
    }).sort_values('abs_coefficient', ascending=False)
    
    top_learned = feature_importance[
        feature_importance['feature'].str.startswith('Feature_')
    ]['feature'].head(10).tolist()
    
    # Analysis 1: Correlations
    print("\n[1/3] Running correlation analysis...")
    corr_results = analyze_feature_correlations(
        df, physics_features, top_learned,
        save_path='physics_vs_learned_correlation.png'
    )
    
    # Analysis 2: Feature Interactions
    print("\n[2/3] Running interaction analysis...")
    interaction_results = analyze_top_learned_features(
        df, physics_features, top_learned[:5],  # Top 5 only
        save_path='feature_explanation.txt'
    )
    
    # Analysis 3: Overlap Analysis (THE KEY CONTRIBUTION)
    print("\n[3/3] Running overlap analysis...")
    top_features_for_overlap = feature_importance['feature'].head(15).tolist()
    overlap_results = analyze_feature_overlap(
        df, top_features_for_overlap,
        save_path='overlap_analysis_key_figure.png'
    )
    
    return {
        'correlations': corr_results,
        'interactions': interaction_results,
        'overlaps': overlap_results
    }


# Load your data
train_latents, val_latents, test_latents, train_labels, val_labels, test_labels = load_data()

# Train your models (your existing code)
lr_model, scaler, y_pred, y_pred_proba = train_logistic_regression(
    train_latents, train_labels, val_latents, val_labels
)

# Run complete analysis
analysis_results = run_complete_analysis(
    train_latents, val_latents, train_labels, val_labels,
    lr_model, scaler
)


Training Logistic Regression...

DATA-FM ANALYSIS PIPELINE
Total samples: 4682
Physics features: 14
Learned features: 71

[1/3] Running correlation analysis...

CORRELATION ANALYSIS: Physics vs Learned Features


/usr/local/lib/python3.11/dist-packages/matplotlib/colors.py:721: RuntimeWarning: invalid value encountered in less
  xa[xa < 0] = -1



Strongest Correlations (|r| > 0.5):

Feature_63:
  entropy: r = -0.528
  HU_p25: r = 0.626
  HU_std: r = -0.575
  mask_frac: r = 0.828

Feature_38:
  HU_p10: r = 0.620

Feature_29:
  HU_p50: r = 0.518
  HU_p75: r = 0.672
  mask_frac: r = -0.512
  homog: r = 0.525

Feature_76:
  entropy: r = 0.571
  HU_p25: r = -0.723
  HU_std: r = 0.652
  mask_frac: r = -0.687

Feature_78:
  HU_p25: r = 0.514
  HU_std: r = -0.500
  mask_frac: r = 0.716
  contrast: r = -0.647

Feature_25:
  HU_p50: r = -0.585
  HU_p25: r = 0.535
  homog: r = -0.520

Feature_33:
  HU_p25: r = 0.637
  mask_frac: r = 0.510

Feature_51:
  mask_frac: r = 0.541

Feature_45:
  entropy: r = -0.595
  HU_p25: r = 0.629
  HU_std: r = -0.604
  mask_frac: r = 0.677

[2/3] Running interaction analysis...

Feature_63:

R² (linear combination):     0.764
R² (with interactions):      0.847
Improvement from interactions: 0.083

Top 5 Contributing Terms:
     term  coefficient  abs_coefficient
mask_frac     0.549613         0.549613
 con